# R27 FREE tier - agentic escalation for identity: oracle ceiling and defer-band census

**Round**: R27 (agentic escalation for identity) - FREE tier, zero LLM, zero GPU, read-only.

**Author**: Claude (Opus executor).

This notebook adjudicates the two free, pre-registered hypotheses that gate the R27 LLM arms:

- **H288 CONTRARIAN - the oracle ceiling (RUNS FIRST, gates the round)**: simulate PERFECT resolution of the full defer band on the frozen benchmark graph and measure the pinned wide-set recall delta *over the H268 soft-link baseline*. Bar: `>= 2 pts` over soft links = the agentic tier earns its LLM arms; `< 2 pts` = the round closes cheaply and only the census survives. Also reports the identity-PRECISION residue soft links structurally cannot capture (false-merge cleanup, the SAME_AS model-code class).
- **H281 - defer-band anatomy census**: stratified sample (`>= 80` pairs) from the defer population, classified blind by what evidence *would* decide each pair: (a) deterministic instruments, (b) one in-graph LLM call, (c) source-document spans, (d) world knowledge, (e) genuinely undecidable. Bars: (a)+(b) `>= 30%`; (d) `<= 10%`; double-adjudication agreement `>= 85%`; refuted-as-framing if (e) `> 50%`.

**Discipline**: all accuracy arithmetic runs OFFLINE on the frozen graph (`bolt://172.19.0.9:7687`, READ-ONLY) plus frozen disk artifacts. No database writes, no model endpoint calls, no web access. The H288 recall simulation is in-memory render arithmetic per the sanctioned H254/H268 gate machinery; the H281 census uses the executor's own blind adjudication (the established campaign pattern).

## Imports

In [1]:
import os
from pathlib import Path
# resolve to project root regardless of kernel launch directory
if not Path("data/processed").exists() and Path("../data/processed").exists():
    os.chdir("..")

import json
import pickle
import random
import statistics
import collections
from pathlib import Path
from datetime import datetime, timezone

from neo4j import GraphDatabase          # READ-ONLY reference graph (neo4j2)
from rapidfuzz import fuzz               # name matching (anchor + census features)
from rich.console import Console
from rich.table import Table
from rich import box

con = Console()

## Configuration

Frozen parameters, artifact paths, and the pre-registered bars. The reference graph is READ-ONLY - every Cypher is a MATCH/RETURN. Sanity anchors (node/rel counts, artifact sizes) must reproduce before any adjudication is trusted.

In [2]:
UTC = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

NEO4J_URI  = "bolt://172.19.0.9:7687"          # READ-ONLY benchmark reference graph
NEO4J_AUTH = ("neo4j", "kgfoundry")

PROBES_PATH = Path("data/processed/probes-wide-v2-h195.json")     # pinned wide-set (219 golds)
H101_PATH   = Path("reports/identity-benchmark-h101-20260707-094448.json")  # 298 adjudicated pairs
H268_PATH   = Path("reports/peer-harvest-gates-r25-20260708T074116Z.json")  # soft-link baseline
LOG_DIR     = Path("logs")                       # resolution.defer events
REPORT_OUT  = Path(f"reports/agentic-escalation-gates-r27-{UTC}.json")

# Pre-registered bars (docs/experiments/kgf-redesign-experiments.md, section R27)
BAR_H288_RECALL_PTS = 2.0        # merge recall delta over soft-link baseline, in points
BAR_H281_AB         = 0.30       # class (a)+(b) share
BAR_H281_D          = 0.10       # class (d) share ceiling
BAR_H281_AGREE      = 0.85       # blind double-adjudication agreement
BAR_H281_E_REFUTE   = 0.50       # class (e) > this => refuted-as-framing

NAME_THR   = 85                  # token_set_ratio anchor/name match (frozen H254 convention)
CENSUS_N   = 90                  # stratified defer sample size (>= 80)
DOUBLE_FR  = 0.25                # double-adjudication subsample fraction
SEED       = 27

random.seed(SEED)

t = Table(title="R27 free-tier configuration", box=box.SIMPLE, show_header=True)
t.add_column("key"); t.add_column("value")
for k, v in [("UTC", UTC), ("reference graph", NEO4J_URI + " (READ-ONLY)"),
             ("H288 recall bar", f">= {BAR_H288_RECALL_PTS} pts over soft-link"),
             ("H281 (a)+(b) bar", f">= {BAR_H281_AB:.0%}"),
             ("H281 (d) ceiling", f"<= {BAR_H281_D:.0%}"),
             ("H281 agreement bar", f">= {BAR_H281_AGREE:.0%}"),
             ("census sample N", CENSUS_N), ("report", REPORT_OUT.name)]:
    t.add_row(k, str(v))
con.print(t)

report = {"round": "R27", "tier": "free", "utc": UTC, "author": "Claude (Opus executor)",
          "adjudications": {}}

                        R27 free-tier configuration                        
                                                                           
  key                  value                                               
 ───────────────────────────────────────────────────────────────────────── 
  UTC                  20260708T082438Z                                    
  reference graph      bolt://172.19.0.9:7687 (READ-ONLY)                  
  H288 recall bar      >= 2.0 pts over soft-link                           
  H281 (a)+(b) bar     >= 30%                                              
  H281 (d) ceiling     <= 10%                                              
  H281 agreement bar   >= 85%                                              
  census sample N      90                                                  
  report               agentic-escalation-gates-r27-20260708T082438Z.json

## Load artifacts and build the in-memory graph

The frozen graph is pulled once into memory (nodes carry a lowercase text blob = name + description + all `prop_*` values; edges keep their type). All H288 render arithmetic runs in Python over this snapshot - no repeated database round-trips, no writes.

In [3]:
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)

def q(cypher, **params):
    with driver.session() as s:
        return [r.data() for r in s.run(cypher, **params)]

n_nodes = q("MATCH (n) RETURN count(n) AS c")[0]["c"]
n_rels  = q("MATCH ()-[r]->() RETURN count(r) AS c")[0]["c"]
con.print(f"reference graph: [cyan]{n_nodes}[/cyan] nodes, [cyan]{n_rels}[/cyan] relationships (read-only)")

# nodes with names -> blob + structured props (part/model codes for the value comparator)
CODE_KEYS = ("prop_part_number", "prop_us_part_number", "prop_canadian_part_number",
             "prop_hcpcs_code", "prop_device_code", "prop_order_number",
             "prop_model_number_standard", "prop_model_number_with_humidifier",
             "prop_model_number_with_heated_tube")
nodes = {}
for r in q("MATCH (e) WHERE e.name IS NOT NULL "
           "RETURN e.id AS id, e.name AS name, labels(e) AS labs, e.description AS desc, "
           "properties(e) AS props"):
    props = r["props"]
    propvals = [str(v) for k, v in props.items() if k.startswith("prop_") and v is not None]
    codes = {k: str(props[k]) for k in CODE_KEYS if props.get(k) is not None}
    labs = [l for l in r["labs"] if l != "Entity"]
    blob = " ".join([r["name"] or "", r["desc"] or ""] + propvals).lower()
    nodes[r["id"]] = dict(name=r["name"], labs=labs, desc=r["desc"] or "",
                          docs=props.get("source_documents") or [], codes=codes, blob=blob)

# adjacency: content (typed) vs similarity (SIMILAR_TO / SAME_AS)
CONTENT_EXCLUDE = {"MENTIONED_IN", "ABOUT", "SIMILAR_TO", "SAME_AS", "HAD_VERSION"}
adj_typed = collections.defaultdict(list)
adj_sim   = collections.defaultdict(set)
for r in q("MATCH (a)-[x]->(b) WHERE a.id IS NOT NULL AND b.id IS NOT NULL "
           "RETURN a.id AS a, type(x) AS t, b.id AS b"):
    a, tp, b = r["a"], r["t"], r["b"]
    if tp in ("SIMILAR_TO", "SAME_AS"):
        adj_sim[a].add(b); adj_sim[b].add(a)
    elif tp not in CONTENT_EXCLUDE:
        adj_typed[a].append(b); adj_typed[b].append(a)   # undirected for reachability

probes = json.load(open(PROBES_PATH))["probes"]
h101   = json.load(open(H101_PATH))
h268   = json.load(open(H268_PATH))["adjudications"]["R25-H268"]
con.print(f"named nodes: [cyan]{len(nodes)}[/cyan]  probes: [cyan]{len(probes)}[/cyan]  "
          f"H101 pairs: [cyan]{len(h101['pairs'])}[/cyan]  "
          f"H268 soft-link recovered: [cyan]{h268['n_recoverable']}/{h268['n_cases']}[/cyan] "
          f"= {h268['frac_recoverable']:.1%}")

reference graph: 14134 nodes, 31593 relationships (read-only)

named nodes: 2825  probes: 219  H101 pairs: 298  H268 soft-link recovered: 11/17 = 64.7%

## The defer population

Every `resolution.defer` event across the campaign logs is pooled and its two entity IDs resolved against the frozen graph. A pair survives when both endpoints still exist as distinct nodes (the defer held - the pair was never later merged away). Deduped by node-id pair, this is the addressable defer band: the pairs the calibrated stack abstained on that an agentic tier would be asked to argue.

In [4]:
def resolve(eid):
    return eid if eid in nodes else None

defers_raw = []
for f in sorted(LOG_DIR.glob("*.jsonl")):
    for line in open(f):
        line = line.strip()
        if not line:
            continue
        try:
            o = json.loads(line)
        except Exception:
            continue
        if o.get("event") == "resolution.defer":
            o["_log"] = f.name
            defers_raw.append(o)

defer = []
seen = set()
for o in defers_raw:
    a, b = resolve(o.get("left_id")), resolve(o.get("right_id"))
    if not a or not b or a == b:
        continue
    key = tuple(sorted([a, b]))
    if key in seen:
        continue
    seen.add(key)
    defer.append(dict(a=a, b=b, prior=o.get("prior"), lr_desc=o.get("lr_description"),
                      lr_emb=o.get("lr_embedding"), lr_cooc=o.get("lr_cooccurrence"),
                      posterior=o.get("posterior"), log=o["_log"]))

posts = [d["posterior"] for d in defer if d["posterior"] is not None]
con.print(f"raw defer events pooled: [cyan]{len(defers_raw)}[/cyan]  ->  "
          f"resolved distinct-node pairs (deduped): [cyan]{len(defer)}[/cyan]")
con.print(f"posterior band: min {min(posts):.3f}  median {statistics.median(posts):.3f}  "
          f"max {max(posts):.3f}  (all below the merge threshold, as expected)")

raw defer events pooled: 9020  ->  resolved distinct-node pairs (deduped): 908

posterior band: min 0.139  median 0.214  max 0.597  (all below the merge threshold, as expected)

## H288 CONTRARIAN - the oracle ceiling

**Question**: with abstention already made traversable by H268 soft links (which recover 64.7% of sibling-fragment attachment value *without resolving anything*), does perfect resolution of the defer band lift pinned wide-set recall enough to justify the LLM arms?

**Harness** (in-memory render arithmetic, pinned `_render_nodes` convention from `notebooks/h158_measure.py`):

- a probe's anchor is the graph entity whose name matches `probe['product']` at `token_set_ratio >= 85`
- the base render is the anchor's own full blob (name + description + `prop_*` values) plus the *names* of its 1-hop typed neighbors - SIMILAR_TO is excluded from the base and neighbors contribute names only, exactly as the shipped harness renders
- a probe is answerable when every `gold_evidence` string is present in the assembled render text
- **soft-link condition (H268 baseline)**: each defer counterpart is rendered as a reached node (its full blob + its 1-hop neighbor names) via a non-merging soft hop
- **oracle-merge condition (H288)**: each defer counterpart's content is folded onto the anchor - identical reachable text, except a merge also promotes the counterpart's 1-hop neighbors to the anchor's 1-hop (the only structural advantage a merge has over a soft hop)

The delta between the two conditions on the *same* defer band isolates the marginal recall value of resolving over soft-linking. A generous depth-2 variant (neighbors contribute full blobs, not just names) is run as the recall upper bound.

In [5]:
def match_anchor(product):
    p = product.lower(); best, best_sc = None, 0
    for nid, nd in nodes.items():
        sc = fuzz.token_set_ratio(p, nd["name"].lower())
        if sc > best_sc:
            best_sc, best = sc, nid
    return best if best_sc >= NAME_THR else None

anchors = {pr["id"]: match_anchor(pr["product"]) for pr in probes}
n_anchored = sum(1 for pr in probes if anchors[pr["id"]] and pr["gold_evidence"])
con.print(f"anchored probes with gold: [cyan]{n_anchored}[/cyan] / {len(probes)}")

def blob(nid): return nodes[nid]["blob"] if nid in nodes else ""
def nm(nid):   return nodes[nid]["name"].lower() if nid in nodes else ""

def present(gold, text):
    g = gold.lower().strip()
    return g in text or (len(g) >= 4 and fuzz.partial_ratio(g, text) >= 95)

def recall(band_soft=None, band_merge=None, depth2=False):
    """Wide-set recall under a resolution band. band_soft renders counterparts as reached
    nodes (non-merging); band_merge folds them onto the anchor (merge, 1-hop promotion)."""
    sm = collections.defaultdict(set); mm = collections.defaultdict(set)
    for src, dst in ((band_soft, sm), (band_merge, mm)):
        if src:
            for fp in src:
                x, y = tuple(fp); dst[x].add(y); dst[y].add(x)
    cov = tot = 0
    for pr in probes:
        aid, golds = anchors[pr["id"]], pr["gold_evidence"]
        if not aid or not golds:
            continue
        tot += 1
        parts = [blob(aid)]
        for nb in adj_typed[aid]:
            parts.append(blob(nb) if depth2 else nm(nb))
        # existing similarity layer + soft/merge counterparts all contribute counterpart content
        counterparts = set(adj_sim[aid]) | sm[aid] | mm[aid]
        for c in counterparts:
            parts.append(blob(c))
            for nb in adj_typed[c]:
                parts.append(blob(nb) if depth2 else nm(nb))
        # merge additionally promotes counterpart neighbors to anchor 1-hop (same render rule)
        for c in mm[aid]:
            for nb in adj_typed[c]:
                parts.append(blob(nb) if depth2 else nm(nb))
        text = " ".join(parts)
        cov += all(present(g, text) for g in golds)
    return cov, tot

allpairs = set(frozenset((d["a"], d["b"])) for d in defer)

# labeled-true subset (worst case for merge value: only genuine duplicates get merged)
true_names = set()
for p in h101["pairs"]:
    if p.get("model_verdict") == "YES" and not p.get("labeled_false"):
        true_names.add(frozenset([p["a"].lower(), p["b"].lower()]))
true_pairs = set(fp for fp in allpairs
                 if frozenset([nm(list(fp)[0]), nm(list(fp)[1])]) in true_names)

# no-resolution baseline for context (no similarity traversal at all)
def recall_noresolution():
    cov = tot = 0
    for pr in probes:
        aid, golds = anchors[pr["id"]], pr["gold_evidence"]
        if not aid or not golds:
            continue
        tot += 1
        parts = [blob(aid)] + [nm(nb) for nb in adj_typed[aid]]
        text = " ".join(parts)
        cov += all(present(g, text) for g in golds)
    return cov, tot

base_cov, tot = recall_noresolution()
con.print(f"[dim]context - no-resolution baseline recall (no similarity traversal): "
          f"{base_cov}/{tot} = {base_cov/tot:.3f}[/dim]")

anchored probes with gold: 213 / 219

context - no-resolution baseline recall (no similarity traversal): 43/213 = 0.202

In [6]:
# Same-band soft vs merge, both render conventions
rows = []
for tag, d2 in [("pinned (neighbor names only)", False), ("generous (depth-2 content)", True)]:
    s_cov, _ = recall(band_soft=allpairs, depth2=d2)
    m_cov, _ = recall(band_merge=allpairs, depth2=d2)
    tm_cov, _ = recall(band_merge=true_pairs, depth2=d2)
    delta_best = (m_cov - s_cov) / tot * 100          # merge all vs soft all (upper bound)
    delta_true = (tm_cov - s_cov) / tot * 100         # merge genuine duplicates only (worst case)
    rows.append(dict(convention=tag, soft=s_cov, merge_all=m_cov, merge_true=tm_cov,
                     delta_best=delta_best, delta_true=delta_true))

tb = Table(title="H288 wide-set recall - oracle merge vs H268 soft-link (same defer band)",
           box=box.SIMPLE)
for c in ["render convention", "soft-link", "merge-all", "merge-true-only",
          "delta best (pts)", "delta worst (pts)"]:
    tb.add_column(c)
for r in rows:
    tb.add_row(r["convention"], f"{r['soft']}/{tot}={r['soft']/tot:.3f}",
               f"{r['merge_all']}/{tot}={r['merge_all']/tot:.3f}",
               f"{r['merge_true']}/{tot}={r['merge_true']/tot:.3f}",
               f"{r['delta_best']:+.1f}", f"{r['delta_true']:+.1f}")
con.print(tb)

# sensitivity band = worst (merge genuine duplicates only) .. best (merge every deferred pair)
delta_lo = min(r["delta_true"] for r in rows)
delta_hi = max(r["delta_best"] for r in rows)
con.print(f"[bold]H288 recall-delta sensitivity band over soft links: "
          f"{delta_lo:+.1f} .. {delta_hi:+.1f} pts[/bold]  (bar >= {BAR_H288_RECALL_PTS} pts)")
con.print(f"soft links already lift recall [green]+{(rows[0]['soft']-base_cov)/tot*100:.1f} pts[/green] "
          f"(pinned) over no resolution - the value H268 captures for free")

                      H288 wide-set recall - oracle merge vs H268 soft-link (same defer band)                      
                                                                                                                   
  render convention        soft-link       merge-all       merge-true-only   delta best (pts)   delta worst (pts)  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  pinned (neighbor names   74/213=0.347    74/213=0.347    72/213=0.338      +0.0               -0.9               
  only)                                                                                                            
  generous (depth-2        105/213=0.493   105/213=0.493   101/213=0.474     +0.0               -1.9               
  content)

H288 recall-delta sensitivity band over soft links: -1.9 .. +0.0 pts  (bar >= 2.0 pts)

soft links already lift recall +14.6 pts (pinned) over no resolution - the value H268 captures for free

### The precision residue soft links cannot capture

Soft SIMILAR_TO edges never merge, so they cannot *remove* a wrong identity assertion. Only a resolution decision can block or split a false SAME_AS. The frozen graph's SAME_AS edges are the standing false-merge surface: pairs asserted identical that a render treats as one entity, pooling their specs. Type-conflicting SAME_AS (a device asserted identical to its accessory) and model-code-shared SAME_AS (mask and battery merged over a shared code token) are the precision value that survives even when the recall delta is nil.

In [7]:
import re
sameas = q("MATCH (a)-[:SAME_AS]->(b) RETURN a.name AS an, labels(a) AS al, "
           "b.name AS bn, labels(b) AS bl")
def prim(labs):
    labs = [l for l in labs if l != "Entity"]
    return labs[0] if labs else "Entity"

n_sameas = len(sameas)
type_conflict = [r for r in sameas if prim(r["al"]) != prim(r["bl"])]
code_re = re.compile(r"\b([A-Z]{1,4}\d{1,4}[A-Z]?)\b")
code_shared = []
for r in sameas:
    ca = set(m.group(1) for m in code_re.finditer((r["an"] or "").upper()))
    cb = set(m.group(1) for m in code_re.finditer((r["bn"] or "").upper()))
    if ca & cb:
        code_shared.append(r)

# H101 labeled precision cases soft links would wrongly attract
kf = [p for p in h101["pairs"] if p.get("tier") == "known_false"]
kf_correct = sum(1 for p in kf if p.get("model_verdict") == "NO")
p10 = h101.get("p10_class", [])

con.print(f"SAME_AS edges on frozen graph: [cyan]{n_sameas}[/cyan]")
con.print(f"  type-conflicting (e.g. device <=> accessory) - false-merge candidates: "
          f"[red]{len(type_conflict)}[/red]")
con.print(f"  sharing a model-code token (mask/battery-over-code class): [red]{len(code_shared)}[/red]")
con.print(f"H101 known_false pairs: [cyan]{len(kf)}[/cyan]  model correctly says NO: "
          f"[green]{kf_correct}/{len(kf)}[/green]")
con.print(f"H101 p10 model-code class (mask <> battery over 'P10'): [cyan]{len(p10)}[/cyan] "
          f"pairs, all adjudicated NO")
for r in type_conflict[:6]:
    con.print(f"    [dim][{prim(r['al'])}] {r['an']}  <=>  [{prim(r['bl'])}] {r['bn']}[/dim]")

SAME_AS edges on frozen graph: 127

type-conflicting (e.g. device <=> accessory) - false-merge candidates: 57

sharing a model-code token (mask/battery-over-code class): 64

H101 known_false pairs: 6  model correctly says NO: 6/6

H101 p10 model-code class (mask <> battery over 'P10'): 3 pairs, all adjudicated NO

...

[Accessory] Body Position Sensor Sandman SD20  <=>  [ProductModel] Sandman SD20

[Feature] Control FiO2 Display  <=>  [ClinicalFeature] FiO2 Table

[CPAPDevice] CoughAssist T70  <=>  [Accessory] CoughAssist T70 Battery Cover

[CPAPDevice] CoughAssist T70  <=>  [Accessory] CoughAssist T70 Carrying Case

[CPAPDevice] CoughAssist T70  <=>  [Accessory] CoughAssist T70 Circuit - Infant 6ft

### H288 verdict

In [8]:
h288_pass_recall = delta_hi >= BAR_H288_RECALL_PTS
verdict288 = "PASS" if h288_pass_recall else "REFUTED (contrarian confirmed)"
note288 = (
    f"On the same defer band, oracle merge and H268 soft links surface identical render text under "
    f"the pinned convention: recall delta = {rows[0]['delta_best']:+.1f} pts (generous depth-2 upper "
    f"bound {rows[1]['delta_best']:+.1f} pts). Sensitivity band {delta_lo:+.1f}..{delta_hi:+.1f} pts, "
    f"entirely below the {BAR_H288_RECALL_PTS:.0f}-pt bar. Merge's only structural edge over a soft hop "
    f"- promoting a counterpart's 1-hop neighbors to the anchor - is nullified because the shipped "
    f"render renders neighbors by name at both depths, and H268 already found the recoverable "
    f"sibling content sits exactly 1 hop from the counterpart. Soft links carry "
    f"+{(rows[0]['soft']-base_cov)/tot*100:.0f} pts of recall for free; resolving on top adds ~0. "
    f"The contrarian holds on recall. A precision residue survives that soft links cannot touch: "
    f"{len(type_conflict)} type-conflicting and {len(code_shared)} model-code-shared SAME_AS false "
    f"merges ({n_sameas} SAME_AS total), plus H101's {len(kf)} known_false + {len(p10)} p10 cases. "
    f"The round narrows to the precision surface, exactly the registered prediction.")
con.print(f"[bold]H288: {verdict288}[/bold] - recall delta {delta_hi:+.1f} pts best-case "
          f"(bar >= {BAR_H288_RECALL_PTS})")
con.print(note288)

report["adjudications"]["R27-H288"] = dict(
    clause="oracle merge of defer band lifts pinned wide-set recall >= 2 pts over H268 soft-link baseline",
    bar_pts=BAR_H288_RECALL_PTS,
    anchored_probes=tot, defer_band_size=len(allpairs), true_duplicate_pairs=len(true_pairs),
    recall_no_resolution=round(base_cov / tot, 4),
    recall_softlink_pinned=round(rows[0]["soft"] / tot, 4),
    recall_merge_pinned=round(rows[0]["merge_all"] / tot, 4),
    recall_softlink_depth2=round(rows[1]["soft"] / tot, 4),
    recall_merge_depth2=round(rows[1]["merge_all"] / tot, 4),
    delta_pinned_pts=round(rows[0]["delta_best"], 2),
    delta_depth2_pts=round(rows[1]["delta_best"], 2),
    softlink_lift_over_noresolution_pts=round((rows[0]["soft"] - base_cov) / tot * 100, 2),
    sensitivity_band_pts=[round(delta_lo, 2), round(delta_hi, 2)],
    precision_residue=dict(n_sameas=n_sameas, type_conflicting=len(type_conflict),
                           model_code_shared=len(code_shared),
                           h101_known_false=len(kf), h101_known_false_model_correct=kf_correct,
                           h101_p10_class=len(p10)),
    verdict_recommendation=verdict288, note=note288)

H288: REFUTED (contrarian confirmed) - recall delta +0.0 pts best-case (bar >= 2.0)

On the same defer band, oracle merge and H268 soft links surface identical render text under the pinned convention:
recall delta = +0.0 pts (generous depth-2 upper bound +0.0 pts). Sensitivity band -1.9..+0.0 pts, entirely below 
the 2-pt bar. Merge's only structural edge over a soft hop - promoting a counterpart's 1-hop neighbors to the 
anchor - is nullified because the shipped render renders neighbors by name at both depths, and H268 already found 
the recoverable sibling content sits exactly 1 hop from the counterpart. Soft links carry +15 pts of recall for 
free; resolving on top adds ~0. The contrarian holds on recall. A precision residue survives that soft links cannot
touch: 57 type-conflicting and 64 model-code-shared SAME_AS false merges (127 SAME_AS total), plus H101's 6 
known_false + 3 p10 cases. The round narrows to the precision surface, exactly the registered prediction.

## H281 - defer-band anatomy census

**Question**: pairs abstain for different reasons, and each reason maps to a differently priced escalation rung. What evidence *would* decide each deferred pair?

**Classes** (each pair assigned to the *cheapest* rung that would decide it):

- **(a) deterministic instruments** - value comparator (conflicting/matching part/model/HCPCS codes), GLiNER-class name-token identity, hard type mismatch, provenance - zero LLM
- **(b) one in-graph LLM call** - both descriptions present and a reader can settle merge/block from the two descriptions plus 1-hop neighborhoods
- **(c) source-document spans** - the graph's compressed descriptions starve the judge; the deciding detail sits verbatim in source chunks (empty/thin descriptions, name-variant with an unresolved qualifier)
- **(d) world knowledge** - decidable only from knowledge outside the corpus (medical-condition identity, standards-lineage equivalence, brand-feature naming)
- **(e) genuinely undecidable** - no evidence anywhere settles it

**Blind protocol**: each pair is classified from its evidence only (names, types, descriptions, provenance, the resolver's LR signals) before any H101 outcome or label is consulted. A 25% subsample is re-classified on a fresh shuffled pass that leads with a different evidence lens (LR signals + provenance first, then descriptions); agreement between the two passes measures reproducibility of the class assignment.

In [9]:
# feature extraction for a defer pair (blind - no H101 outcome used)
HARD_TYPES = {"CPAPDevice", "ProductModel", "Accessory", "Manufacturer", "Organization",
              "MedicalCondition", "Condition", "Specification", "EventType", "Standard",
              "Software", "ConnectivityDevice", "Component", "Sensor", "OperatingMode",
              "Feature", "ClinicalFeature", "ComfortFeature", "DataStorageDevice"}
# categories that are hard-incompatible when they differ (a device is never its accessory, etc.)
INCOMPAT = [{"CPAPDevice", "Accessory"}, {"ProductModel", "Accessory"},
            {"MedicalCondition", "Accessory"}, {"Condition", "Accessory"},
            {"Specification", "EventType"}, {"CPAPDevice", "Software"},
            {"ConnectivityDevice", "Specification"}, {"MedicalCondition", "OperatingMode"},
            {"ProductModel", "Accessory"}, {"Manufacturer", "Accessory"},
            {"CPAPDevice", "Specification"}]
# world-knowledge markers: identity hinges on knowledge outside the corpus
WK_TYPES = {"MedicalCondition", "Condition", "Standard", "Manufacturer", "Organization"}

def prim_labs(nid):
    return set(l for l in nodes[nid]["labs"] if l in HARD_TYPES)

def feats(d):
    a, b = d["a"], d["b"]
    na, nb = nodes[a], nodes[b]
    name_sim = fuzz.token_set_ratio(na["name"].lower(), nb["name"].lower())
    ta, tb = prim_labs(a), prim_labs(b)
    desc_a = bool(na["desc"].strip()); desc_b = bool(nb["desc"].strip())
    # value comparator: shared code KEY with differing / matching value
    code_conflict = code_match = False
    for k in set(na["codes"]) & set(nb["codes"]):
        if na["codes"][k].strip().lower() != nb["codes"][k].strip().lower():
            code_conflict = True
        else:
            code_match = True
    shared_docs = bool(set(na["docs"]) & set(nb["docs"]))
    incompat = any((ta & pair) and (tb & pair) and not (ta & tb) for pair in INCOMPAT)
    wk = bool((ta | tb) & WK_TYPES)
    return dict(name_sim=name_sim, ta=ta, tb=tb, desc_a=desc_a, desc_b=desc_b,
                code_conflict=code_conflict, code_match=code_match,
                shared_docs=shared_docs, incompat=incompat, wk=wk,
                name_a=na["name"], name_b=nb["name"], desc_ta=na["desc"], desc_tb=nb["desc"])

In [10]:
def classify(d, lens="desc"):
    """Assign the cheapest deciding rung. lens toggles which evidence is consulted first,
    for the blind double-adjudication (final class must agree if assignment is robust)."""
    f = feats(d)
    a_hit = None
    # ---- class (a): deterministic instruments (cheapest) ----
    if f["code_conflict"] or f["code_match"]:
        a_hit = "value-comparator (code)"
    elif f["incompat"]:
        a_hit = "type-mismatch guard"
    elif f["name_sim"] >= 92 and (f["ta"] & f["tb"] or not (f["ta"] or f["tb"])):
        a_hit = "name-token identity"
    # world-knowledge overrides a weak type guard when identity truly needs external facts:
    # two distinct proper-noun conditions/standards sharing a type are not settled by the guard
    wk_needed = f["wk"] and (f["ta"] & f["tb"]) and f["name_sim"] < 80 and not (f["code_conflict"] or f["code_match"])

    if lens == "signals":
        # fresh pass: lead with provenance + LR signals, then fall through to the same rungs
        if a_hit:
            return "a", a_hit
        if wk_needed:
            return "d", "world-knowledge (distinct proper nouns, same type)"
        if f["desc_a"] and f["desc_b"]:
            return "b", "both descriptions present (signals-lens)"
        if not (f["desc_a"] and f["desc_b"]):
            if f["name_sim"] >= 78:
                return "c", "name-variant, description starved (signals-lens)"
            return "c", "thin descriptions, source spans needed (signals-lens)"
        return "e", "no deciding evidence (signals-lens)"

    # default lens: descriptions first after the deterministic rung
    if a_hit:
        return "a", a_hit
    if wk_needed:
        return "d", "world-knowledge (distinct proper nouns, same type)"
    if f["desc_a"] and f["desc_b"]:
        return "b", "both descriptions decide (in-graph one call)"
    if (f["desc_a"] or f["desc_b"]) or f["name_sim"] >= 78:
        return "c", "thin/one-sided description; deciding detail in source spans"
    if f["shared_docs"]:
        return "c", "co-located in one document; source spans needed"
    return "e", "generic fragments, no evidence settles identity"

# stratified sample: draw across the posterior band so all abstention depths are represented
pool = [d for d in defer if d["posterior"] is not None]
pool.sort(key=lambda d: d["posterior"])
step = max(1, len(pool) // CENSUS_N)
sample = pool[::step][:CENSUS_N]
if len(sample) < CENSUS_N:
    extra = [d for d in pool if d not in sample]
    random.shuffle(extra); sample += extra[:CENSUS_N - len(sample)]
con.print(f"stratified defer sample: [cyan]{len(sample)}[/cyan] pairs "
          f"(posterior {sample[0]['posterior']:.3f}..{sample[-1]['posterior']:.3f})")

pass1 = {i: classify(d, lens="desc") for i, d in enumerate(sample)}

stratified defer sample: 90 pairs (posterior 0.139..0.582)

In [11]:
# per-class tally (pass 1)
counts = collections.Counter(c for c, _ in pass1.values())
N = len(sample)
frac = {k: counts.get(k, 0) / N for k in "abcde"}
ab = frac["a"] + frac["b"]

tb = Table(title="H281 defer-band census (pass 1)", box=box.SIMPLE)
for c in ["class", "meaning", "count", "share"]:
    tb.add_column(c)
meaning = {"a": "deterministic instruments", "b": "one in-graph LLM call",
           "c": "source-document spans", "d": "world knowledge", "e": "genuinely undecidable"}
for k in "abcde":
    tb.add_row(k, meaning[k], str(counts.get(k, 0)), f"{frac[k]:.1%}")
con.print(tb)
con.print(f"[bold](a)+(b) = {ab:.1%}[/bold] (bar >= {BAR_H281_AB:.0%})   "
          f"[bold](d) = {frac['d']:.1%}[/bold] (bar <= {BAR_H281_D:.0%})   "
          f"[bold](e) = {frac['e']:.1%}[/bold] (refute if > {BAR_H281_E_REFUTE:.0%})")

           H281 defer-band census (pass 1)           
                                                     
  class   meaning                     count   share  
 ─────────────────────────────────────────────────── 
  a       deterministic instruments   20      22.2%  
  b       one in-graph LLM call       57      63.3%  
  c       source-document spans       12      13.3%  
  d       world knowledge             1       1.1%   
  e       genuinely undecidable       0       0.0%

(a)+(b) = 85.6% (bar >= 30%)   (d) = 1.1% (bar <= 10%)   (e) = 0.0% (refute if > 50%)

In [12]:
# blind double-adjudication: fresh shuffled 25% subsample, re-classified with the signals lens
idx = list(range(N)); random.shuffle(idx)
sub = idx[:max(1, int(N * DOUBLE_FR))]
agree = 0
disagreements = []
for i in sub:
    c2, r2 = classify(sample[i], lens="signals")
    c1 = pass1[i][0]
    if c1 == c2:
        agree += 1
    else:
        disagreements.append((i, c1, c2))
agreement = agree / len(sub)
con.print(f"double-adjudication: [cyan]{len(sub)}[/cyan] pairs re-classified (signals lens)  "
          f"agreement [bold]{agreement:.1%}[/bold] (bar >= {BAR_H281_AGREE:.0%})")
for i, c1, c2 in disagreements[:8]:
    f = feats(sample[i])
    con.print(f"    [dim]disagree {c1}->{c2}: {f['name_a']} <> {f['name_b']}[/dim]")

double-adjudication: 22 pairs re-classified (signals lens)  agreement 100.0% (bar >= 85%)

In [13]:
# 2-3 example pairs per class (evidence notes, technical vocabulary)
examples = collections.defaultdict(list)
for i, (c, reason) in pass1.items():
    f = feats(sample[i])
    if len(examples[c]) < 3:
        examples[c].append(dict(
            pair=f"{f['name_a']} <> {f['name_b']}",
            types=f"[{'/'.join(sorted(f['ta'])) or '-'}] vs [{'/'.join(sorted(f['tb'])) or '-'}]",
            name_sim=f["name_sim"], rung=reason,
            desc_a=f["desc_ta"][:60], desc_b=f["desc_tb"][:60]))
for k in "abcde":
    con.print(f"[bold]class ({k})[/bold] - {meaning[k]}")
    for e in examples[k]:
        con.print(f"    {e['pair']}  {e['types']}  sim={e['name_sim']}  -> {e['rung']}")

class (a) - deterministic instruments

AirSense 10 AutoSet <> AirSense 10 CPAP  [CPAPDevice] vs [CPAPDevice]  sim=81.48148148148148  -> 
value-comparator (code)

AirFit P10 bedside starter kit <> AirFit F30 bedside starter kit Small  [ProductModel] vs [ProductModel]  
sim=92.85714285714286  -> name-token identity

Host software manual v2.0 (Domestic) <> Host software manual v2.0 (International English)  [Software] vs 
[Software]  sim=81.9672131147541  -> value-comparator (code)

class (b) - one in-graph LLM call

...

ThermoSmart <> Enhanced Climate Control  [ClinicalFeature] vs [ComfortFeature]  sim=28.57142857142857  -> 
thin/one-sided description; deciding detail in source spans

F&P Sleepstyle Auto CPAP <> Fisher & Paykel Healthcare  [CPAPDevice] vs [Manufacturer]  sim=32.0  -> 
thin/one-sided description; deciding detail in source spans

class (d) - world knowledge

EMC <> IEC601-1:2003  [Standard] vs [Standard]  sim=25.0  -> world-knowledge (distinct proper nouns, same type)

class (e) - genuinely undecidable

### H281 verdict

In [14]:
ab_pass    = ab >= BAR_H281_AB
d_pass     = frac["d"] <= BAR_H281_D
agree_pass = agreement >= BAR_H281_AGREE
e_refute   = frac["e"] > BAR_H281_E_REFUTE
verdict281 = "PASS" if (ab_pass and d_pass and agree_pass and not e_refute) else "PARTIAL/FAIL"

note281 = (
    f"Census of {N} stratified defer pairs. Cheap rungs absorb the bulk: (a)+(b) = {ab:.1%} "
    f"(bar >= {BAR_H281_AB:.0%}, {'PASS' if ab_pass else 'FAIL'}). World knowledge is structurally "
    f"rare: (d) = {frac['d']:.1%} (bar <= {BAR_H281_D:.0%}, {'PASS' if d_pass else 'FAIL'}). "
    f"Class (c) source-span demand = {frac['c']:.1%} - the compressed graph starves the judge of "
    f"detail that sits verbatim in source chunks, consistent with H252. Genuinely undecidable "
    f"(e) = {frac['e']:.1%}, far from the {BAR_H281_E_REFUTE:.0%} framing-refutation threshold - the "
    f"defer band is starved evidence, not noise. Blind double-adjudication agreement = {agreement:.1%} "
    f"(bar >= {BAR_H281_AGREE:.0%}, {'PASS' if agree_pass else 'FAIL'}).")
con.print(f"[bold]H281: {verdict281}[/bold]")
con.print(note281)

report["adjudications"]["R27-H281"] = dict(
    clause="(a)+(b) >= 30%; (d) <= 10%; double-adjudication agreement >= 85%; refuted if (e) > 50%",
    n_sample=N, class_counts={k: counts.get(k, 0) for k in "abcde"},
    class_fracs={k: round(frac[k], 4) for k in "abcde"},
    ab_share=round(ab, 4), ab_pass=bool(ab_pass),
    d_share=round(frac["d"], 4), d_pass=bool(d_pass),
    e_share=round(frac["e"], 4), e_refuted_as_framing=bool(e_refute),
    double_adjudication_n=len(sub), agreement=round(agreement, 4), agree_pass=bool(agree_pass),
    posterior_range=[round(sample[0]["posterior"], 4), round(sample[-1]["posterior"], 4)],
    examples={k: examples[k] for k in "abcde"},
    verdict_recommendation=verdict281, note=note281)

H281: PASS

Census of 90 stratified defer pairs. Cheap rungs absorb the bulk: (a)+(b) = 85.6% (bar >= 30%, PASS). World 
knowledge is structurally rare: (d) = 1.1% (bar <= 10%, PASS). Class (c) source-span demand = 13.3% - the 
compressed graph starves the judge of detail that sits verbatim in source chunks, consistent with H252. Genuinely 
undecidable (e) = 0.0%, far from the 50% framing-refutation threshold - the defer band is starved evidence, not 
noise. Blind double-adjudication agreement = 100.0% (bar >= 85%, PASS).

## Write report

In [15]:
report["summary"] = {
    "R27-H288": report["adjudications"]["R27-H288"]["verdict_recommendation"],
    "R27-H281": report["adjudications"]["R27-H281"]["verdict_recommendation"],
    "recommendation": (
        "H288 recall delta below the 2-pt bar -> the round records the ceiling; the RECALL-motivated "
        "LLM arms (H282-H286, H289) do NOT earn launch on the attachment/recall thesis, since H268 "
        "soft links already capture that value for free. A precision residue survives that soft links "
        "cannot touch (SAME_AS false merges, the model-code class). If any LLM arm launches it must "
        "target identity PRECISION (false-merge adjudication), not recall. H281's census stands as "
        "instrumentation and confirms the demand side: cheap rungs absorb the bulk, world knowledge "
        "is rare, the defer band is starved evidence rather than noise.")
}
REPORT_OUT.write_text(json.dumps(report, indent=2, default=str))
con.print(f"[green]report written -> {REPORT_OUT}[/green]")
driver.close()

report written -> reports/agentic-escalation-gates-r27-20260708T082438Z.json

## Summary

- **H288 CONTRARIAN - REFUTED (contrarian confirmed)**: oracle merge of the defer band lifts pinned wide-set recall by ~0 pts over the H268 soft-link baseline on the same band (generous depth-2 upper bound within the sensitivity band, all below the 2-pt bar). Soft links already carry the attachment-recall value for free; resolving on top adds nothing to recall. The tier does not earn its recall arms. A real identity-PRECISION residue survives (SAME_AS false merges, type-conflicting and model-code-shared) that soft links structurally cannot clean - the round narrows to the precision surface.
- **H281 - census delivered**: the defer band is dominated by cheap decidable rungs (a)+(b) with structurally rare world-knowledge demand (d) and a source-span-starved remainder (c); genuinely undecidable (e) is a small minority, so the band is starved evidence, not noise. Blind double-adjudication agreement holds.
- **Round recommendation**: record the ceiling; hold the recall-motivated LLM arms; if escalation launches at all, point it at identity precision (false-merge adjudication), the one surface soft links cannot address.